# `Mf6Tid` — Transient Idomain for MODFLOW 6

This notebook demonstrates `Mf6Tid` and `Mf6TidRunner` from `flopy.mf6.tid`.

**Two components:**
- `Mf6Tid` — pure Python schedule object. Build, inspect, serialise. No MF6 binary needed.
- `Mf6TidRunner` — BMI glue. Applies the schedule to a running MF6 via `modflowapi`.

**Important runtime constraint:**  
The DIS/DISV package must have `idomain=1` for every cell that will ever be activated — MF6 excludes `idomain=0` cells from its internal node system at initialisation time and they cannot be recovered later.  
Use `tid.add(kper=0, deactivate=[...])` to mark cells that should start inactive.

---
**Requirements:**
- `flopy` (this fork)
- `modflowapi` — only for `Mf6TidRunner`: `pip install modflowapi`
- MF6 shared library (`libmf6.dylib` / `.so` / `.dll`) at the path set in `LIB_PATH` below

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import flopy
from flopy.mf6 import (
    MFSimulation, ModflowGwf, ModflowGwfdis, ModflowGwfic,
    ModflowGwfnpf, ModflowGwfchd, ModflowGwfoc, ModflowIms, ModflowTdis,
    Mf6Tid, Mf6TidRunner,
)
from flopy.utils import HeadFile

# ── Path to the MF6 shared library (edit as needed) ──────────────────────────
# Repo-local bin/ — self-contained fork
LIB_PATH = Path(__file__).parent / "bin" / "libmf6.dylib" if "__file__" in dir() else Path.cwd() / "bin" / "libmf6.dylib"
# Override if needed:
# LIB_PATH = Path("/usr/local/lib/libmf6.so")   # Linux
# LIB_PATH = Path(r"C:\mf6\mf6.dll")          # Windows
# ─────────────────────────────────────────────────────────────────────────────

print("flopy version:", flopy.__version__)
print("lib exists:", LIB_PATH.exists())

## Part 1 — `Mf6Tid` schedule object (no binary needed)

Build a schedule, inspect `idomain_at()`, write and reload the sidecar JSON.

In [ ]:
# ── Minimal model (no IMS, no run — just enough for FloPy to resolve nodes) ──
ws_unit = Path(tempfile.mkdtemp())
sim_u = MFSimulation(sim_name="mfsim", sim_ws=str(ws_unit))
ModflowTdis(sim_u, nper=10, perioddata=[(1.0, 1, 1.0)] * 10)
gwf_u = ModflowGwf(sim_u, modelname="gwf")

# 3×3×1 grid, center cell (0,1,1) starts inactive in DIS
idomain = np.ones((1, 3, 3), dtype=int)
idomain[0, 1, 1] = 0
ModflowGwfdis(gwf_u, nlay=1, nrow=3, ncol=3, idomain=idomain)

tid = Mf6Tid(gwf_u)

# Schedule: activate center at kper=2, deactivate at kper=5, re-activate at kper=7
tid.add(kper=2, activate=[(0, 1, 1)])
tid.add(kper=5, deactivate=[(0, 1, 1)])
tid.add(kper=7, activate=[(0, 1, 1)])

print("Schedule:", tid.schedule)

In [ ]:
# ── Inspect idomain_at() ─────────────────────────────────────────────────────
center_node = gwf_u.modelgrid.get_node((0, 1, 1))[0]
print(f"Center node (flat index): {center_node}")
print()

print(f"{'kper':>5}  {'center idomain':>15}  state")
print("-" * 35)
for k in range(10):
    v = tid.idomain_at(k)[center_node]
    state = "ACTIVE" if v == 1 else "inactive"
    print(f"{k:>5}  {v:>15}  {state}")

In [ ]:
# ── Visualise idomain_at() ───────────────────────────────────────────────────
center_vals = [tid.idomain_at(k)[center_node] for k in range(10)]

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.step(range(10), center_vals, where="post", lw=2, color="steelblue")
ax.fill_between(range(10), center_vals, step="post", alpha=0.25, color="steelblue")
ax.set_xlabel("Stress period (kper)")
ax.set_ylabel("idomain")
ax.set_title("Center cell (0,1,1) — Mf6Tid schedule")
ax.set_yticks([0, 1])
ax.set_yticklabels(["0  inactive", "1  active"])
ax.set_xticks(range(10))
ax.grid(axis="x", ls=":")
plt.tight_layout()
plt.show()

In [ ]:
# ── Write / load round-trip ──────────────────────────────────────────────────
sim_u.write_simulation(silent=True)   # creates workspace so json path exists
tid.write()

json_path = ws_unit / "gwf.tid.json"
print("JSON written to:", json_path)
print(json_path.read_text())

In [ ]:
# Reload and verify
tid2 = Mf6Tid.load(gwf_u)
print("Reloaded schedule:", tid2.schedule)

match = all(
    np.array_equal(tid.idomain_at(k), tid2.idomain_at(k))
    for k in range(10)
)
print("Round-trip idomain_at() identical:", match)

## Part 2 — `Mf6TidRunner` (requires `modflowapi` + MF6 shared library)

3×3×1 DIS model. Left column CHD=1, right column CHD=0.  
The centre column cell (0,1,1) starts inactive and is toggled by the schedule.

In [ ]:
# ── Build the model ──────────────────────────────────────────────────────────
NPER = 10
ws = Path(tempfile.mkdtemp())
print("Workspace:", ws)

sim = MFSimulation(sim_name="mfsim", sim_ws=str(ws))
# 10 time steps per SP ensures steady-state convergence each period
ModflowTdis(sim, nper=NPER, perioddata=[(10.0, 10, 1.0)] * NPER)
ModflowIms(sim, complexity="MODERATE")

gwf = ModflowGwf(sim, modelname="gwf", save_flows=True)

# ALL cells idomain=1 — required by runner
ModflowGwfdis(gwf, nlay=1, nrow=3, ncol=3)
ModflowGwfic(gwf, strt=1.0)
ModflowGwfnpf(gwf, icelltype=0, k=1.0)

# CHD: left col head=1, right col head=0
chd_data = [([(0, r, 0), 1.0]) for r in range(3)] + [([(0, r, 2), 0.0]) for r in range(3)]
ModflowGwfchd(gwf, stress_period_data={0: chd_data})

ModflowGwfoc(
    gwf,
    head_filerecord="gwf.hds",
    budget_filerecord="gwf.cbb",
    saverecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

sim.write_simulation(silent=True)
print("Files written:", [f.name for f in ws.iterdir()])

In [ ]:
# ── Build the TID schedule ───────────────────────────────────────────────────
tid_run = Mf6Tid(gwf)

# Center cell deactivated from kper=0 ("inactive from day one")
tid_run.add(kper=0, deactivate=[(0, 1, 1)])
# Activate at kper=2
tid_run.add(kper=2, activate=[(0, 1, 1)])
# Deactivate again at kper=5
tid_run.add(kper=5, deactivate=[(0, 1, 1)])
# Re-activate at kper=7 (tests round-trip re-activation)
tid_run.add(kper=7, activate=[(0, 1, 1)])

tid_run.write()
print("TID schedule written to:", ws / "gwf.tid.json")

# Preview the schedule
center = gwf.modelgrid.get_node((0, 1, 1))[0]
print()
print(f"{'kper':>5}  {'center':>8}  expected head")
print("-" * 35)
for k in range(NPER):
    v = tid_run.idomain_at(k)[center]
    note = "active → ~0.5" if v == 1 else "inactive → frozen"
    print(f"{k:>5}  {'ACTIVE' if v else 'inactive':>8}  {note}")

In [ ]:
# ── Run via BMI ──────────────────────────────────────────────────────────────
runner = Mf6TidRunner(sim, tid_run, lib_path=LIB_PATH)
runner.run()
print("Run complete.")

In [ ]:
# ── Read heads ───────────────────────────────────────────────────────────────
hf = HeadFile(str(ws / "gwf.hds"))
heads = hf.get_alldata()         # shape (NPER, nlay, nrow, ncol)
flat  = heads.reshape(NPER, 9)   # flat node ordering

print("Head array shape:", heads.shape)
print()
print(f"{'kper':>5}  {'center head':>12}  {'left-center':>12}  {'right-center':>13}")
print("-" * 50)
for k in range(NPER):
    # node 3=(0,1,0), 4=(0,1,1), 5=(0,1,2)
    print(f"{k:>5}  {flat[k,4]:>12.4f}  {flat[k,3]:>12.4f}  {flat[k,5]:>13.4f}")

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
kpers = np.arange(NPER)

# Idomain state from schedule (for background shading)
active = np.array([tid_run.idomain_at(k)[center] for k in kpers])

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
# Shade inactive SPs
for k in kpers:
    if active[k] == 0:
        ax.axvspan(k - 0.5, k + 0.5, color="lightcoral", alpha=0.25, zorder=0)

node_labels = {
    3: ("left-center (0,1,0) CHD=1", "#2196F3"),
    4: ("CENTER (0,1,1) — TID controlled", "#E91E63"),
    5: ("right-center (0,1,2) CHD=0", "#4CAF50"),
    1: ("top-center (0,0,1)", "#FF9800"),
    7: ("bot-center (0,2,1)", "#9C27B0"),
}
for node, (label, color) in node_labels.items():
    lw = 2.5 if node == 4 else 1.2
    ls = "-" if node == 4 else "--"
    ax.plot(kpers, flat[:, node], marker="o", ms=5, lw=lw, ls=ls,
            color=color, label=label)

ax.set_ylabel("Head (m)")
ax.set_title("Mf6TidRunner — 3×3×1 DIS model, 10 SPs\n"
             "Red shading = center cell inactive (IBOUND=0)")
ax.legend(fontsize=8, loc="upper right")
ax.set_ylim(-0.1, 1.2)
ax.grid(ls=":")

ax2 = axes[1]
ax2.step(kpers, active, where="mid", lw=2, color="#E91E63")
ax2.fill_between(kpers, active, step="mid", alpha=0.3, color="#E91E63")
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["0 inactive", "1 active"])
ax2.set_xlabel("Stress period (kper)")
ax2.set_ylabel("Center\nIBOUND")
ax2.set_xticks(kpers)
ax2.grid(ls=":")

plt.tight_layout()
plt.savefig(ws / "Mf6Tid_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to:", ws / "Mf6Tid_demo.png")

In [ ]:
# ── Verify key assertions (same as autotest) ─────────────────────────────────
IC_HEAD = 1.0

# kper=0,1: inactive → head frozen at IC
for k in [0, 1]:
    assert abs(flat[k, center] - IC_HEAD) < 1e-6, f"kper={k}: expected IC head"

# kper=2,3,4: active → converges to 0.5
for k in [2, 3, 4]:
    assert abs(flat[k, center] - 0.5) < 0.01, f"kper={k}: expected 0.5"

# kper=5,6: inactive → head frozen at last solved value
assert abs(flat[5, center] - flat[4, center]) < 1e-8, "kper=5: head not frozen"
assert abs(flat[6, center] - flat[4, center]) < 1e-8, "kper=6: head not frozen"

# kper=7,8,9: re-activated → recovers to 0.5
for k in [7, 8, 9]:
    assert abs(flat[k, center] - 0.5) < 0.01, f"kper={k}: expected 0.5 after re-activation"

print("All assertions passed.")

## Part 3 — DISV grid

`Mf6Tid` works identically for DISV — node identifiers are `(lay, cell2d)` tuples.

In [ ]:
# ── 1-layer, 4-cell quad DISV (no binary run — just schedule + node resolution)
ws_v = Path(tempfile.mkdtemp())
sim_v = MFSimulation(sim_name="mfsim", sim_ws=str(ws_v))
ModflowTdis(sim_v, nper=5, perioddata=[(1.0, 1, 1.0)] * 5)
gwf_v = ModflowGwf(sim_v, modelname="gwf")

from flopy.mf6 import ModflowGwfdisv
vertices = [
    [0, 0.0, 0.0], [1, 1.0, 0.0], [2, 2.0, 0.0],
    [3, 0.0, 1.0], [4, 1.0, 1.0], [5, 2.0, 1.0],
    [6, 0.0, 2.0], [7, 1.0, 2.0], [8, 2.0, 2.0],
]
cell2d = [
    [0, 0.5, 0.5, 4, 0, 1, 4, 3],
    [1, 1.5, 0.5, 4, 1, 2, 5, 4],
    [2, 0.5, 1.5, 4, 3, 4, 7, 6],
    [3, 1.5, 1.5, 4, 4, 5, 8, 7],
]
ModflowGwfdisv(gwf_v, nlay=1, ncpl=4, vertices=vertices, cell2d=cell2d,
               top=1.0, botm=[0.0])

mg_v = gwf_v.modelgrid
print("Grid type:", mg_v.grid_type)
print("nnodes:", mg_v.nnodes)

# Build schedule with (lay, cell2d) tuples
tid_v = Mf6Tid(gwf_v)
tid_v.add(kper=0, deactivate=[(0, 2), (0, 3)])  # top-right cells start inactive
tid_v.add(kper=2, activate=[(0, 2)])             # activate cell 2 at kper=2
tid_v.add(kper=4, activate=[(0, 3)])             # activate cell 3 at kper=4

print()
print("DISV schedule:", tid_v.schedule)
print()

# Check node resolution agrees with modelgrid
for tup in [(0, 0), (0, 1), (0, 2), (0, 3)]:
    expected = mg_v.get_node(tup)[0]
    print(f"  get_node{tup} = {expected}")

print()
print(f"{'kper':>5}  {'cell2 idomain':>14}  {'cell3 idomain':>14}")
print("-" * 40)
for k in range(5):
    arr = tid_v.idomain_at(k)
    print(f"{k:>5}  {arr[2]:>14}  {arr[3]:>14}")

## Part 4 — Re-loading the schedule from JSON

The `.tid.json` sidecar survives `write_simulation()` / `load()` round-trips.

In [ ]:
# Write the runner model's TID, reload, verify
tid_run.write()

tid_reload = Mf6Tid.load(gwf)   # gwf is the 3×3×1 DIS model from Part 2

print("Reloaded schedule keys:", sorted(tid_reload.schedule.keys()))
print("kper=0 deactivate:", tid_reload.schedule[0]["deactivate"])
print("kper=2 activate  :", tid_reload.schedule[2]["activate"])
print()

match = all(
    np.array_equal(tid_run.idomain_at(k), tid_reload.idomain_at(k))
    for k in range(NPER)
)
print("idomain_at() identical after reload:", match)

## Summary

| Capability | Works without binary | Works with modflowapi |
|---|---|---|
| `Mf6Tid.add()` — schedule transitions | ✓ | ✓ |
| `Mf6Tid.idomain_at(kper)` — cumulative state | ✓ | ✓ |
| `Mf6Tid.write()` / `Mf6Tid.load()` — JSON sidecar | ✓ | ✓ |
| DIS `(lay,row,col)` node tuples | ✓ | ✓ |
| DISV `(lay,cell2d)` node tuples | ✓ | ✓ |
| DISU flat-int node IDs | ✓ | ✓ |
| `Mf6TidRunner.run()` — BMI execution | — | ✓ |

**Key rule for `Mf6TidRunner`:**  
Write your FloPy model with `idomain=1` for all cells that will ever be used.  
Use `tid.add(kper=0, deactivate=[...])` to specify cells that start inactive.